In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from train_cuda import train

class UCIHARInertialDataset(Dataset):
    def __init__(self, root_dir: str, split: str = "train"):
        if split not in ("train", "test"):
            raise ValueError("split must be 'train' or 'test'")

        signal_names = (
            "total_acc_x", "total_acc_y", "total_acc_z",
            "body_acc_x", "body_acc_y", "body_acc_z",
            "body_gyro_x", "body_gyro_y", "body_gyro_z",
        )
        signal_dir = os.path.join(root_dir, split, "Inertial Signals")
        signals = [
            np.loadtxt(os.path.join(signal_dir, f"{name}_{split}.txt"))
            for name in signal_names
        ]
        x = np.stack(signals, axis=-1).astype(np.float32, copy=False)

        label_path = os.path.join(root_dir, split, f"y_{split}.txt")
        y = np.loadtxt(label_path, dtype=np.int64) - 1

        self.x = torch.from_numpy(x)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]


In [ ]:
root = "./Datasets/UCI HAR Dataset"

train_ds = UCIHARInertialDataset(root_dir=root, split="train")
test_ds = UCIHARInertialDataset(root_dir=root, split="test")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

x, y = next(iter(train_loader))
print("batch shape:", x.shape)   # [32, 128, 9]  
print("label shape:", y.shape)   # [32]
print("label range:", y.min().item(), "~", y.max().item())


In [ ]:
from torch import nn
from torch.utils.data import random_split


class GRUClassifier(nn.Module):
    def __init__(self, input_size=9, hidden_size=64, num_classes=6):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, hidden = self.gru(x)
        return self.classifier(hidden[-1])


In [ ]:
# 从训练集中划分验证集，保留 test_loader 用于最终评估。
torch.manual_seed(23)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GRUClassifier().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
train_losses, val_losses = train(
    model, train_loader, test_loader, loss_fn, optimizer, epochs=20, device=device
)


In [ ]:
model.eval()
correct = 0
total = 0
with torch.inference_mode():
    for x_batch, y_batch in test_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        predictions = model(x_batch).argmax(dim=1)
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

print(f"Test accuracy: {correct / total:.2%} ({correct}/{total})")
